# Scaling Karşılaştırması — Word2Vec

Word2Vec vektörleştirme üzerine 8 farklı ölçekleme yönteminin etkisi.
Her scaler × her sınıflandırıcı kombinasyonu çalıştırılır ve karşılaştırmalı sonuçlar üretilir.

### Kütüphaneler

In [1]:
import pandas as pd
import numpy as np
import time
import copy
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
import xgboost as xgb


### Veri Yükleme ve Bölme

In [2]:
df = pd.read_csv('spam_cleaned.csv', encoding='latin-1')

X, y = df['sms'], df['type']
print(f"Veri seti: {len(df)} satır | ham: {(y==0).sum()}, spam: {(y==1).sum()}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=111, stratify=y
)
print(f"Eğitim: {len(X_train)} | Test: {len(X_test)} | Train spam oranı: {y_train.mean()*100:.1f}%")


Veri seti: 5570 satır | ham: 4823, spam: 747
Eğitim: 3899 | Test: 1671 | Train spam oranı: 13.4%


### Word2Vec Vektörleştirme

In [3]:
import gensim.downloader as api
print("Word2Vec modeli yükleniyor (word2vec-google-news-300)...")
w2v_model = api.load("word2vec-google-news-300")
print("Yüklendi.")

def sms_to_vec(text, model, dim=300):
    tokens = str(text).lower().split()
    vecs = [model[w] for w in tokens if w in model]
    return np.mean(vecs, axis=0) if vecs else np.zeros(dim)

X_train_vec = np.array([sms_to_vec(t, w2v_model) for t in X_train])
X_test_vec  = np.array([sms_to_vec(t, w2v_model) for t in X_test])
X_train_vec = np.abs(X_train_vec)
X_test_vec  = np.abs(X_test_vec)
print(f"Word2Vec özellik boyutu: {X_train_vec.shape[1]}")


Word2Vec modeli yükleniyor (word2vec-google-news-300)...
Yüklendi.
Word2Vec özellik boyutu: 300


### Değerlendirme Fonksiyonu

In [4]:
import scipy.stats as stats

def evaluate(name, clf, X_tr, y_tr, X_te, y_te):
    t0 = time.time()
    clf.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    y_pred = clf.predict(X_te)
    try:
        y_score = clf.predict_proba(X_te)[:, 1]
    except AttributeError:
        y_score = clf.decision_function(X_te)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(clf, X_tr, y_tr, cv=cv, scoring='accuracy')
    cv_acc = cv_scores.mean()
    cv_std = cv_scores.std()
    ci = stats.t.interval(0.95, df=len(cv_scores)-1, loc=cv_acc, scale=stats.sem(cv_scores))
    return {
        'Model':        name,
        'Accuracy':     round(accuracy_score(y_te, y_pred), 4),
        'Precision':    round(precision_score(y_te, y_pred, zero_division=0), 4),
        'Recall':       round(recall_score(y_te, y_pred), 4),
        'F1':           round(f1_score(y_te, y_pred), 4),
        'ROC-AUC':      round(roc_auc_score(y_te, y_score), 4),
        'CV-Acc':       round(cv_acc, 4),
        'CV-Std':       round(cv_std, 4),
        'CI-95% Low':   round(ci[0], 4),
        'CI-95% High':  round(ci[1], 4),
        'Time(s)':      round(elapsed, 4),
    }

### Tüm Scaler'lar × Tüm Sınıflandırıcılar

8 farklı ölçekleme yöntemi sırayla uygulanır ve sonuçlar tek tabloda karşılaştırılır:
StandardScaler, MinMaxScaler, RobustScaler, Normalizer, Binarizer, PowerTransformer, QuantileTransformer, MaxAbsScaler

In [6]:
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, RobustScaler,
                                    Normalizer, Binarizer, PowerTransformer,
                                    QuantileTransformer, MaxAbsScaler)

scalers = {
    'StandardScaler':      StandardScaler(),
    'MinMaxScaler':        MinMaxScaler(),
    'RobustScaler':        RobustScaler(),
    'Normalizer':          Normalizer(norm='l2'),
    'Binarizer':           Binarizer(threshold=0.0),
    'PowerTransformer':    PowerTransformer(method='yeo-johnson'),
    'QuantileTransformer': QuantileTransformer(output_distribution='normal', random_state=42),
    'MaxAbsScaler':        MaxAbsScaler(),
}

classifiers = [
    ('LR',  LogisticRegression(max_iter=1000)),
    ('DT',  DecisionTreeClassifier(random_state=42)),
    ('KNN', KNeighborsClassifier(n_neighbors=3)),
    ('SVM', SVC(kernel='linear', probability=True)),
    ('NC',  NearestCentroid()),
]

all_rows = []
total = len(scalers) * len(classifiers)
done  = 0

for scaler_name, scaler in scalers.items():
    X_tr_s = scaler.fit_transform(X_train_vec)
    X_te_s = scaler.transform(X_test_vec)
    for clf_name, clf in classifiers:
        t0 = time.time()
        row = evaluate(clf_name, copy.deepcopy(clf), X_tr_s, y_train, X_te_s, y_test)
        row['Scaler'] = scaler_name
        all_rows.append(row)
        done += 1
        elapsed = time.time() - t0
        print(f"[{done:2d}/{total}] {scaler_name:22s} + {clf_name:3s}  "
              f"Acc={row['Accuracy']:.4f}  F1={row['F1']:.4f}  ({elapsed:.1f}s)")
    print()

cols = ['Scaler', 'Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'CV-Acc', 'CV-Std', 'CI-95% Low', 'CI-95% High', 'Time(s)']
df_results = pd.DataFrame(all_rows)[cols]
print(df_results.to_string(index=False))


[ 1/40] StandardScaler         + LR   Acc=0.9755  F1=0.9083  (0.1s)
[ 2/40] StandardScaler         + DT   Acc=0.9348  F1=0.7594  (4.9s)
[ 3/40] StandardScaler         + KNN  Acc=0.9701  F1=0.8904  (0.1s)
[ 4/40] StandardScaler         + SVM  Acc=0.9671  F1=0.8802  (2.9s)
[ 5/40] StandardScaler         + NC   Acc=0.9587  F1=0.8535  (0.1s)

[ 6/40] MinMaxScaler           + LR   Acc=0.9797  F1=0.9209  (0.0s)
[ 7/40] MinMaxScaler           + DT   Acc=0.9348  F1=0.7594  (4.9s)
[ 8/40] MinMaxScaler           + KNN  Acc=0.9695  F1=0.8894  (0.1s)
[ 9/40] MinMaxScaler           + SVM  Acc=0.9767  F1=0.9108  (2.7s)
[10/40] MinMaxScaler           + NC   Acc=0.9557  F1=0.8452  (0.0s)

[11/40] RobustScaler           + LR   Acc=0.9737  F1=0.9009  (0.1s)
[12/40] RobustScaler           + DT   Acc=0.9348  F1=0.7594  (4.9s)
[13/40] RobustScaler           + KNN  Acc=0.9713  F1=0.8947  (0.1s)
[14/40] RobustScaler           + SVM  Acc=0.9665  F1=0.8783  (3.0s)
[15/40] RobustScaler           + NC   Acc=0.95